# PyCaret - SHAP - LIME

### Libraries

In [ ]:
# Pin compatible numpy/pandas versions required by PyCaret
!pip uninstall -y numpy pandas
!pip install --only-binary=:all: numpy==1.26.4 pandas==2.1.4

In [ ]:
# Import all PyCaret regression functions into the global namespace
from pycaret.regression import *

In [ ]:
import pandas as pd
import numpy as np

print("PyCaret environment ready.")

### Load Dataset

In [ ]:
# Read the csv cleaned 
df = pd.read_csv("../Data/cars_cleaned.csv")
df.head()

In [ ]:
# Remove fe_timestamp
if "fe_timestamp" in df.columns:
    df = df.drop(columns=["fe_timestamp"])

df.head()

In [ ]:
# Check dimensions and structure
df.shape, df.columns

In [ ]:
# Descriptive statistics
df.describe().T

In [ ]:
# Check null values
df.isna().sum()

In [ ]:
# Check duplicates
df.duplicated().sum()

## PyCaret - SHAP - Log

### PyCaret

#### Prepare the columns

In [ ]:
# Features + log_price target for the log-scale model
cols_to_use = [
    'make',
    'engine_displacement_cc',
    'battery_capacity_kwh',
    'acc_0_100_min',
    'acc_0_100_max',
    'acc_0_100_mean',
    'acc_0_100_is_range',
    'acc_0_100_is_instant',
    'is_commercial',
    'horsepower_min',
    'horsepower_max',
    'horsepower_mean',
    'horsepower_is_range',
    'top_speed_kmh',
    'seats',
    'torque_nm',
    'is_ev',
    'is_hybrid',
    'is_phev',
    'fuel_macro',
    'segment',
    'is_luxury_brand',
    'is_performance_luxury_brand',
    'is_premium_brand',
    'acceleration_class',
    'hp_class',
    'performance_score',
    'acc_missing_flag',
    'log_price'
]

In [ ]:
# Subset dataframe to selected features
df_model = df[cols_to_use].copy()
df_model.head()

In [ ]:
# Structural fill with 0: domain logic, not statistical imputation (no leakage risk)
# battery_capacity_kwh = 0 for non-EVs, engine_displacement_cc = 0 for EVs
structural_features = ["engine_displacement_cc", "battery_capacity_kwh"]
df_model[structural_features] = df_model[structural_features].fillna(0)

# Numerical and categorical imputation is handled by PyCaret after the train/test split
print("Nulls remaining:", df_model.isna().sum().sum())

#### PyCaret Setup

In [ ]:
df_model['_price_bins'] = pd.qcut(
    df['price_usd'], q=10, labels=False, duplicates='drop'
).values
print(f"Price bins: {df_model['_price_bins'].nunique()} bins")
print(df_model['_price_bins'].value_counts().sort_index())

In [ ]:
# Setup PyCaret — median/mode imputation applied after split, no leakage
reg = setup(
    data=df_model,
    target="log_price",
    session_id=3,
    train_size=0.8,
    numeric_imputation='median',
    categorical_imputation='mode',
    data_split_stratify=['_price_bins'],
    ignore_features=['_price_bins'],
    preprocess=True,
    normalize=False,
    verbose=False
)

In [ ]:
import joblib, os
_nb_dir = os.path.dirname(os.path.abspath("07_pyCaret.ipynb"))

X_test = get_config('X_test')
y_test = get_config('y_test')

joblib.dump(X_test, os.path.join(_nb_dir, "X_test_pycaret_log.pkl"))
joblib.dump(y_test, os.path.join(_nb_dir, "y_test_pycaret_log.pkl"))
print("Saved to:", _nb_dir)

#### Model Comparison

In [ ]:
# Train and rank all available regression models via cross-validation
best = compare_models()
display(best)

**Note**: PyCaret evaluated a wide range of regression models on the car pricing dataset using cross-validation. The results show that **gradient boosting ensemble models** lead, with **LightGBM** as the top performer (R² ≈ 0.93), closely followed by **Extra Trees** and **Random Forest**. All three achieve MAE ≈ 0.14–0.18 on the log scale, indicating strong and stable predictive accuracy.

Linear models and distance-based approaches perform significantly worse on this dataset, confirming the presence of non-linear relationships and interaction effects.

**Conclusion:**  
LightGBM and Random Forest are selected as candidates for the blended ensemble (dynamic deduplication: top-1 model per metric: R², RMSE, MAE).

### Blender

In [ ]:
best_r2   = compare_models(n_select=1, sort='R2')
best_rmse = compare_models(n_select=1, sort='RMSE')
best_mae  = compare_models(n_select=1, sort='MAE')

seen = {}
for m in [best_r2, best_rmse, best_mae]:
    name = _base_name(m)
    if name not in seen:
        seen[name] = m

log_models = list(seen.values())
log_labels = list(seen.keys())
print("LOG blend models:", log_labels)

In [ ]:
def _base_name(pc_model):
    try:
        return type(pc_model.named_steps['actual_estimator']).__name__
    except (AttributeError, KeyError):
        return type(pc_model).__name__

if not isinstance(best_models, list):
    best_models = [best_models]

log_models = best_models
log_labels = [_base_name(m) for m in log_models]
print("LOG top models:", log_labels)

In [ ]:
if len(log_models) == 1:
    blender = log_models[0]
    print("Single model — blender = best model directly")
else:
    blender = blend_models(estimator_list=log_models)
blender

In [ ]:
# Export blender CV metrics — log scale
metrics_blender_log = pull()
metrics_blender_log.to_csv("../Data/metrics_pycaret_blender_log.csv", index=True)
metrics_blender_log

In [ ]:
# Required for PyCaret's interactive evaluate_model widget
!pip install anywidget

In [ ]:
# Interactive model diagnostics (residuals, QQ, feature importance, etc.)
evaluate_model(blender)

### Feature Importance Plot

In [ ]:
def _plot_feature_importance(pc_model, label):
    try:
        est = pc_model.named_steps['actual_estimator']
    except (AttributeError, KeyError):
        est = pc_model
    if hasattr(est, 'feature_importances_'):
        print(f"Feature importance — {label}")
        plot_model(pc_model, plot='feature')
    else:
        print(f"Feature importance — {label}: not available, skipping.")

for m, lbl in zip(log_models, log_labels):
    _plot_feature_importance(m, lbl)

### Save PyCaret LOG models

In [ ]:
save_model(best, "../Models/pycaret_best_log")
save_model(blender, "../Models/pycaret_blender_log")

for i, m in enumerate(log_models, start=1):
    save_model(m, f"../Models/pycaret_log_model_{i}")

print("PyCaret LOG models saved:", log_labels)

### SHAP

In [ ]:
import shap  # Model explainability library

In [ ]:
# Retrieve the transformed training set from PyCaret's pipeline
X_train = get_config("X_train_transformed")

In [ ]:
# Draw random background samples for the SHAP kernel explainer
X_sample_200 = shap.utils.sample(X_train, 200, random_state=3)
X_sample_500 = shap.utils.sample(X_train, 500, random_state=3)

In [ ]:
# Build model-agnostic SHAP explainers using the blender's predict function
explainer_200 = shap.Explainer(blender.predict, X_sample_200)
explainer_500 = shap.Explainer(blender.predict, X_sample_500)

In [ ]:
# Compute SHAP values for each sample
shap_values_200 = explainer_200(X_sample_200)
shap_values_500 = explainer_500(X_sample_500)

In [ ]:
# Global feature impact overview (n=200)
shap.summary_plot(shap_values_200, X_sample_200)

**Note**: The SHAP summary plot provides a global view of how the blended model (LightGBM + Random Forest) uses the transformed features to generate price predictions on the log scale. Each point represents a single car in the training sample, with color indicating whether the feature value is low (blue) or high (red), and the horizontal position showing whether that feature pushes the prediction upward or downward.

The model relies primarily on performance‑related variables such as horsepower (min/mean/max), torque, engine displacement, and top speed. Higher values of these features consistently increase the predicted price, confirming that the model captures the expected relationship between vehicle performance and market value.

Brand‑related indicators (e.g., premium, luxury, performance brands) and segment categories (Sportscar, Supercar) also have a strong positive impact on predicted price, reflecting the influence of brand positioning and vehicle class. Conversely, features associated with low‑power configurations tend to reduce the predicted value.

Overall, the SHAP analysis shows that the model's behavior is coherent, stable, and aligned with domain knowledge: performance, brand, and segment characteristics are the dominant drivers of price, while secondary attributes contribute with smaller, but interpretable, effects.

In [ ]:
# Global feature impact overview (n=500)
shap.summary_plot(shap_values_500, X_sample_500)

**Note**: The SHAP summary plot provides a global view of how the blended model (LightGBM + Random Forest) uses the transformed features to generate price predictions on the log scale. Each point represents a single car in the training sample, with color indicating whether the feature value is low (blue) or high (red), and the horizontal position showing whether that feature pushes the prediction upward or downward.

The model relies primarily on performance‑related variables such as horsepower (min/mean/max), torque, engine displacement, and top speed. Higher values of these features consistently increase the predicted price, confirming that the model captures the expected relationship between vehicle performance and market value.

Brand‑related indicators (premium, luxury, performance brands) and segment categories (Sportscar, Supercar) also have a strong positive impact on predicted price, reflecting the influence of brand positioning and vehicle class. Conversely, features associated with low‑power configurations tend to reduce the predicted value.

Overall, the SHAP analysis shows that the model's behavior is coherent, stable, and aligned with domain knowledge: performance, brand, and segment characteristics are the dominant drivers of price, while secondary attributes contribute with smaller, but interpretable, effects.

In [ ]:
# the waterfall_plot shows how we use each variable to make a better prediction (sample = 500)
sample_ind = 0
shap.plots.waterfall(shap_values_500[sample_ind], max_display=14)

**Note**: The SHAP waterfall plot explains how the model arrives at the predicted price for
this specific car. The visualization starts from the **base value** (the average
prediction across all training samples) and then adds the contribution of each
feature, either increasing or decreasing the final output.

High-performance attributes such as horsepower (min/mean/max), torque, engine
displacement, and top speed contribute strongly and positively to the prediction,
pushing the estimated price upward. Segment indicators (Supercar) and
power‑class features (Extreme) also add substantial positive impact, reflecting
the vehicle’s high-performance profile.

Conversely, features with low or zero values for premium or performance brand
indicators contribute little or negatively, slightly reducing the prediction. The
final value at the bottom of the plot represents the model’s predicted price for
this specific record after aggregating all feature contributions.

Overall, the waterfall plot provides a transparent, step-by-step breakdown of how
each feature influences the model’s decision for this individual car.

In [ ]:
# Mean absolute SHAP values — global feature ranking
shap.plots.bar(shap_values_500)

**Note**: The SHAP bar plot provides a clear ranking of the features that most influence the
model’s price predictions. Each bar represents the average absolute contribution of
a feature across all predictions, making this visualization ideal for business
stakeholders who want to understand the key drivers of the model without technical
complexity.

Performance-related attributes dominate the model’s logic. Minimum and average
horsepower are the strongest predictors, followed by torque and engine
displacement. These features consistently have the largest impact on the estimated
price, confirming that the model heavily relies on technical performance to assess
vehicle value.

Brand information also plays a significant role, reflecting how market perception
and brand positioning influence pricing. Segment indicators such as “Supercar”
further contribute to higher predicted values, aligning with real-world automotive
pricing dynamics.

Overall, the bar plot highlights the core factors that shape the model’s decisions:
performance, brand, and vehicle segment. These insights provide a transparent and
business-friendly summary of what the model has learned about car pricing.

## PyCaret - SHAP - LIME USD

### PyCaret

In [ ]:
# Features + price_usd target for the interpretability-focused model
cols_to_use_usd = [
    'make',
    'engine_displacement_cc',
    'battery_capacity_kwh',
    'acc_0_100_min',
    'acc_0_100_max',
    'acc_0_100_mean',
    'acc_0_100_is_range',
    'acc_0_100_is_instant',
    'is_commercial',
    'horsepower_min',
    'horsepower_max',
    'horsepower_mean',
    'horsepower_is_range',
    'top_speed_kmh',
    'seats',
    'torque_nm',
    'is_ev',
    'is_hybrid',
    'is_phev',
    'fuel_macro',
    'segment',
    'is_luxury_brand',
    'is_performance_luxury_brand',
    'is_premium_brand',
    'acceleration_class',
    'performance_score',
    'acc_missing_flag',
    'hp_class',
    'price_usd'
]

df_model_usd = df[cols_to_use_usd].copy()

In [ ]:
# Structural fill with 0: domain logic, not statistical imputation (no leakage risk)
# battery_capacity_kwh = 0 for non-EVs, engine_displacement_cc = 0 for EVs
structural_features = ["engine_displacement_cc", "battery_capacity_kwh"]
df_model_usd[structural_features] = df_model_usd[structural_features].fillna(0)

# Numerical and categorical imputation is handled by PyCaret after the train/test split
print("Nulls remaining:", df_model_usd.isna().sum().sum())

In [ ]:
df_model_usd['_price_bins'] = pd.qcut(
    df_model_usd['price_usd'], q=10, labels=False, duplicates='drop'
)
print(f"Price bins: {df_model_usd['_price_bins'].nunique()} bins")
print(df_model_usd['_price_bins'].value_counts().sort_index())

In [ ]:
# Re-initialize PyCaret with USD target (overwrites the log-scale session)
# Median/mode imputation applied after split — no leakage
reg_usd = setup(
    data=df_model_usd,
    target="price_usd",
    session_id=3,
    train_size=0.8,
    numeric_imputation='median',
    categorical_imputation='mode',
    data_split_stratify=['_price_bins'],
    ignore_features=['_price_bins'],
    preprocess=True,
    normalize=False,
    verbose=False
)

In [ ]:
X_test = get_config('X_test')
y_test = get_config('y_test')

joblib.dump(X_test, os.path.join(_nb_dir, "X_test_pycaret_usd.pkl"))
joblib.dump(y_test, os.path.join(_nb_dir, "y_test_pycaret_usd.pkl"))
print("Saved to:", _nb_dir)

In [ ]:
# Compare models on raw USD price
best_usd = compare_models()
best_usd

**Note**: PyCaret evaluated the same set of regression models on raw USD prices. The ranking shifts substantially compared to the log-scale run. The top performers are **Huber Regressor** (best overall), followed by **LightGBM** and **Random Forest**. Overall performance is significantly lower than the log-scale counterpart, driven by the heavy right tail of the price distribution and the inherent difficulty of predicting raw USD prices across a range spanning from budget vehicles to million-dollar supercars.

These three models are selected for blending purely to support **SHAP and LIME interpretability** on the USD scale, so feature contributions are expressed directly in dollars without back-transformation. The weaker aggregate metrics are an expected trade-off, not a model failure.

### Blender

In [ ]:
best_r2   = compare_models(n_select=1, sort='R2')
best_rmse = compare_models(n_select=1, sort='RMSE')
best_mae  = compare_models(n_select=1, sort='MAE')

seen = {}
for m in [best_r2, best_rmse, best_mae]:
    name = _base_name(m)
    if name not in seen:
        seen[name] = m

usd_models = list(seen.values())
usd_labels = list(seen.keys())
print("USD blend models:", usd_labels)

In [ ]:
if len(usd_models) == 1:
    blender_usd = usd_models[0]
    print("Single model — blender_usd = best model directly")
else:
    blender_usd = blend_models(estimator_list=usd_models)
blender_usd

In [ ]:
# Export blender CV metrics — USD scale
metrics_blender_usd = pull()
metrics_blender_usd.to_csv("../Data/metrics_pycaret_blender_usd.csv", index=True)
metrics_blender_usd

In [ ]:
# Interactive model diagnostics (residuals, QQ, feature importance, etc.)
evaluate_model(blender_usd)

**Note**: The blended model (Huber + LightGBM + Random Forest) on raw USD prices yields a mean **R² ≈ 0.51** and very high RMSE, significantly weaker than the log-scale counterpart (R² ≈ 0.92). This is expected: raw USD prices are extremely skewed and the top models on this scale are not pure tree-based ensembles but include robust regression approaches (Huber), which struggle with the extreme right tail of the distribution. Despite the weak aggregate metrics, the model's SHAP and LIME attributions remain valid for interpretability purposes: feature contributions are expressed directly in dollars, making the model's reasoning immediately accessible without any back-transformation.

### Feature Importance Plot

In [ ]:
for m, lbl in zip(usd_models, usd_labels):
    _plot_feature_importance(m, lbl)

### Save PyCaret USD models

In [ ]:
save_model(best_usd, "../Models/pycaret_best_usd")
save_model(blender_usd, "../Models/pycaret_blender_usd")

for i, m in enumerate(usd_models, start=1):
    save_model(m, f"../Models/pycaret_usd_model_{i}")

print("PyCaret USD models saved:", usd_labels)

### SHAP

In [ ]:
# Retrieve the transformed training set from the USD PyCaret pipeline
X_train_usd = get_config("X_train_transformed")

In [ ]:
# Draw random background samples for the SHAP kernel explainer
X_sample_usd_200 = shap.utils.sample(X_train_usd, 200, random_state=3)
X_sample_usd_500 = shap.utils.sample(X_train_usd, 500, random_state=3)

In [ ]:
# Build model-agnostic SHAP explainers — predictions in USD
explainer_usd_200 = shap.Explainer(blender_usd.predict, X_sample_usd_200)
explainer_usd_500 = shap.Explainer(blender_usd.predict, X_sample_usd_500)

In [ ]:
# Compute SHAP values — contributions expressed in USD
shap_values_usd_200 = explainer_usd_200(X_sample_usd_200)
shap_values_usd_500 = explainer_usd_500(X_sample_usd_500)

In [ ]:
# Global feature impact overview — USD contributions (n=200)
shap.summary_plot(shap_values_usd_200, X_sample_usd_200)

**Note**: The SHAP summary plot (n=200) shows feature contributions expressed directly in **USD**. Each point represents a single car: color indicates feature value (blue = low, red = high), and horizontal position shows the dollar impact on the predicted price.

The x-axis range is now much wider than in the log-scale version: a single feature can shift the prediction by tens or even hundreds of thousands of dollars for extreme vehicles. Horsepower (min/mean), torque, and engine displacement are the dominant drivers, with high values pushing the predicted price upward by significant dollar amounts. Brand indicators (luxury, performance) and segment (Supercar) also contribute large positive shifts, directly reflecting their market premium.

This representation is the most intuitive for a business audience: rather than log-units, every contribution is in the same currency as the price itself.

In [ ]:
# Global feature impact overview — USD contributions (n=500)
shap.summary_plot(shap_values_usd_500, X_sample_usd_500)

**Note**: With 500 samples the picture is more stable and the feature ranking consolidates. The spread of SHAP values confirms that the model captures a very wide price range: budget cars cluster near the base value with small contributions in either direction, while high-performance and luxury vehicles accumulate large positive SHAP values from multiple correlated features (horsepower, torque, segment, brand). The consistency between n=200 and n=500 validates that the explainer has converged and the results are not sample-dependent.

In [ ]:
# Waterfall plot: step-by-step USD price decomposition for a single car (sample = 500)
sample_ind = 0
shap.plots.waterfall(shap_values_usd_500[sample_ind], max_display=14)

**Note**: The SHAP waterfall plot decomposes the predicted price of a single car into individual dollar contributions. Starting from the base value (average predicted price across training), each feature either adds or subtracts dollars until reaching the final prediction.

This is the most interpretable output for a non-technical audience: instead of abstract log-units or percentage shifts, the plot directly answers *"why does this car cost $X?"*, for example, *"the Supercar segment adds $80,000, high horsepower adds $35,000, but the absence of a luxury brand flag reduces the estimate by $12,000."*

The cumulative nature of the plot makes it ideal for slide presentations, where each step can be narrated to walk stakeholders through the model's reasoning.

In [ ]:
# Mean absolute SHAP values in USD — global feature ranking
shap.plots.bar(shap_values_usd_500)

**Note**: The SHAP bar plot ranks features by their average absolute dollar contribution across all predictions. Each bar represents how many dollars, on average, a feature shifts the predicted price away from the base value, making this the most business-friendly summary of model behavior.

Horsepower (min and mean), torque, and engine displacement dominate, consistent with the log-scale results. The key advantage over the log-scale version is **quantification**: we can now say, for instance, that horsepower is on average responsible for a ±$X,000 shift in predicted price, giving the model's logic an immediately actionable interpretation for pricing strategy or market analysis.

### LIME

In [ ]:
!pip install lime

In [ ]:
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt

# Build LIME explainer on the USD training set (fitted only on training data)
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_usd.values,
    feature_names=X_train_usd.columns.tolist(),
    mode='regression',
    random_state=3
)

In [ ]:
# Select 3 representative instances: budget, mid-range, luxury
import numpy as np

# Wrapper to convert LIME's numpy arrays back to DataFrame (preserves column names for PyCaret)
def predict_fn(data):
    return blender_usd.predict(pd.DataFrame(data, columns=X_train_usd.columns))

preds_all = predict_fn(X_train_usd.values)
sorted_idx = np.argsort(preds_all)

idx_budget  = sorted_idx[len(sorted_idx) // 10]       # ~10th percentile
idx_mid     = sorted_idx[len(sorted_idx) // 2]        # ~50th percentile
idx_luxury  = sorted_idx[int(len(sorted_idx) * 0.90)] # ~90th percentile

for label, idx in [("Budget", idx_budget), ("Mid-range", idx_mid), ("Luxury", idx_luxury)]:
    exp = lime_explainer.explain_instance(
        X_train_usd.values[idx],
        predict_fn,
        num_features=10
    )
    print(f"\n--- {label} car | predicted: ${preds_all[idx]:,.0f} ---")
    exp.as_pyplot_figure()
    plt.title(f"LIME Explanation — {label} Car")
    plt.tight_layout()
    plt.show()

**Note**: LIME (Local Interpretable Model-agnostic Explanations) fits a local linear model around each individual prediction, approximating the blended model's behavior in the neighborhood of that specific car. Unlike SHAP (which is globally consistent and additively exact), LIME is a local approximation, making it ideal for explaining single decisions in plain language.

Three representative cars are explained across the full price spectrum:

**Budget car (~10th percentile)**: LIME highlights low horsepower and non-premium brand as the main factors keeping the price low. The model correctly associates these features with the economy segment.

**Mid-range car (~50th percentile)**: The explanation shows a balanced profile: moderate horsepower and a mainstream brand contribute positively, while the absence of luxury indicators limits the upside.

**Luxury car (~90th percentile)**: High horsepower, luxury or performance brand flags, and a premium segment drive the price sharply upward. The local linear model captures these interactions with high-magnitude coefficients.

LIME and SHAP are complementary: SHAP provides the global picture of what the model has learned, while LIME provides the local, car-by-car story; together they give a complete and presentation-ready interpretability framework.